# SV3 Data Intermediate Preprocessing 

This notebook runs a user through the steps to select a campaign and preprocess data beginning with RINEX data in our EarthScope Jupyter Hub. It is specific to the steps for processing SV3 data. 

If you wish to begin with novatel files and generate your own RINEX files, then use the example scripts in `es_sfgtools`. 

* Note: This notebook can take a while to run through (specifically generating positions with pride). You may also use the example scripts for intermediate processing on your local machine if preferred. 

## Import modules
Here we will import the modules we need to run through the notebook. Our `WorkflowHandler` contains all of the necessary code to run the intermediate processing steps. 

In [ ]:
import os
from pathlib import Path
import json
import subprocess

from es_sfgtools.workflows.workflow_handler import WorkflowHandler

### Browse available campaigns from the community archive and select target

Locate the campaign of interest in https://gage-data.earthscope.org/archive/seafloor, and note the `network`, `station`, and `campaign` names, which will be input in the cell below.  

- Note: the cascadia-gorda raw data is currently hidden (by request) but still usable, here are the available campaigns

    |  | GCC1 | NBR1 | NCC1 |
    |---|---|---|---|
    | **2022** |2022_A_1065 | 2022_A_1065  |  2022_A_1065 |
    | **2023** |  2023_A_1063 | 2023_A_1063 | 2023_A_1063 |
    | **2024** |  2024_A_1126 |  2024_A_1126 | 2024_A_1126 |


*In order to use this notebook to process new campaigns, the data must first be submitted and made available from the community archive*

In [ ]:
# Input survey parameters
network='cascadia-gorda'
site='NCC1'
campaign='2025_A_1126'

# Set data directory path for local environment
data_dir = Path(f"{os.path.expanduser('~/data/sfg')}")
raw_data_dir = data_dir / network / site / campaign / "raw"

#### USE THE FOLLOWING DEFAULTS UNLESS DESIRED ####
os.makedirs(data_dir, exist_ok=True)
workflow = WorkflowHandler(directory=data_dir)
workflow.set_network_station_campaign(network_id=network, 
                                      station_id=site, 
                                      campaign_id=campaign)
                                      
print(f"Workflow directory: {workflow.directory}")
print(f"Raw data directory for campaign: {raw_data_dir}")

## Optional Steps - ingest raw data or download raw data from the cloud
### Option 1: Ingest Local Raw Data

If you already have raw data files downloaded on your this machine, use this option to register them with the workflow catalog.

The code below will scan the `raw_data_dir` directory (defined above) and add any existing raw data files to the internal catalog, making them available for processing. This is useful when you've manually downloaded files or are reusing data from a previous session.

In [ ]:
# Ingest raw data from the local raw data directory
workflow.ingest_add_local_data(raw_data_dir)

### Option 2: Download Data from Community Archive

If you don't have data downloaded locally, this option retrieves raw data from the EarthScope community archive (https://gage-data.earthscope.org/archive/seafloor).

**Two-step process:**
1. **Catalog the available data**: `workflow.ingest_catalog_archive_data()` queries the archive and creates an inventory of available files for your selected campaign

2. **Download the data**: `workflow.ingest_download_intermediate_archive_data()` downloads the necessary data files to your local directory. For intermediate, we skip over the raw novatel files that are used to conver to rinex and direclty download the rinex files and other file that will be used in this pipeline.
    * Choose whether to download 1hz rinex or higher rate data by using the variable `rinex_1Hz` equal to `True` or `False`. If true 1Hz rinex will be chosen over higher rate data. This may be preferable for processing in the hub due to storage or memory for running `pride ppp`.  

The workflow automatically identifies and downloads the appropriate file types. Files already present locally are skipped.

In [ ]:
# 1. Ingest catalog data
workflow.ingest_catalog_archive_data()

In [ ]:
workflow.ingest_download_intermediate_archive_data(rinex_1Hz=False)

## Configure Processing Parameters

The `global_config` dictionary below defines settings for each stage of the data processing pipeline. These settings control how raw data is converted into GARPOS-ready inputs.

### Configuration Sections:

- **`dfop00_config`**: Settings for processing acoustic ping/reply sequences from DFOP00 files
  - `override`: Set to `True` to reprocess even if data already exists

- **`position_update_config`**: Settings for interpolating waveglider positions to ping times
  - `lengthscale`: Gaussian process lengthscale parameter for smoothing (in seconds)
  - `plot`: Set to `True` to generate diagnostic plots
  - `override`: Set to `True` to reprocess existing data

- **`pride_config`**: Settings for PRIDE-PPPAR precise point positioning
  - `cutoff_elevation`: Minimum satellite elevation angle (degrees)
  - `system`: GNSS constellation(s) - "GREC23J" = GPS/GLONASS/Galileo/BDS/QZSS
  - `frequency`: GNSS frequency bands to use for each system
  - `loose_edit`: Use relaxed editing for high-dynamic waveglider data
  - `sample_frequency`: Output position sampling rate (Hz)
  - `tides`: Tide corrections - "SOP" = solid/ocean/polar
  - `override`: Set to `True` to reprocess existing solutions
  - `override_products_download`: Set to `True` to re-download orbit/clock products

- **`rinex_config`**: Settings for generating positions from RINEX
  - `n_processes`: Number of parallel processes for RINEX PPP processing
  - `time_interval`: Length of each RINEX file in hours (24 = daily files)
  - `override`: Set to `True` to regenerate existing RINEX files

Unused in this intermediate pipline - changing these variables will not affect the current pipeline
- **`novatel_config`**: Settings for processing GNSS range observations from Novatel receivers
  - `n_processes`: Number of parallel processes to use (adjust based on your CPU cores)
  - `override`: Set to `True` to reprocess existing data

**Note**: Set `override=False` to skip processing steps where outputs already exist. This is useful for resuming interrupted workflows.

In [ ]:
global_config = {
    "dfop00_config": {
        "override": True
        },
    "position_update_config": {
        "override": True,
        "lengthscale": 0.1,
        "plot": False
        },
    "pride_config": {
        "cutoff_elevation": 7,
        "end": None,
        "frequency": ["G12", "R12", "E15", "C26", "J12"],
        "high_ion": None,
        "interval": None,
        "local_pdp3_path": None,
        "loose_edit": True,
        "sample_frequency": 1,
        "start": None,
        "system": "GREC23J",
        "tides": "SOP",
        "override_products_download": False,
        "override": False
        },
    "rinex_config": {
        "n_processes": 5,
        "time_interval": 24,
        "override": False
        },
    "novatel_config": {
        "n_processes": 5,
        "override": False
        }
    } 

## Run the Intermediate SV3 Processing Pipeline

This single command executes the entire data processing workflow, transforming raw SV3 data into GARPOS-ready observation files.

**What it does:**

1. `run_pride`: Runs PRIDE-PPP on RINEX files to generate KIN and residual files.
    1. Retrieves RINEX files needing processing
    2. Downloads GNSS product files (SP3, OBX, ATT) for each unique DOY
    3. Runs PRIDE-PPPAR in parallel to convert RINEX to KIN format
    4. Adds KIN and residual files to asset catalog

2. `process_kinematic`: Processes KIN files to generate kinematic position dataframes.
    1. Retrieves KIN files needing processing
    2. Converts each KIN file to a structured dataframe
    3. Writes dataframes to kinematic position TileDB array
    4. Marks files as processed in asset catalog

3. `process_dfop00`: Processes Sonardyne DFOP00 files to generate preliminary shotdata.
    1. Retrieves DFOP00 files needing processing
    2. Converts each file to shotdata dataframe (acoustic ping-reply
        sequences)
    3. Writes dataframes to preliminary shotdata TileDB array
    4. Marks files as processed in asset catalog

4. `update_shotdata`: Refines shotdata with interpolated high-precision kinematic positions. This step significantly improves position accuracy by replacing GNSS
    positions with interpolated PRIDE-PPP solutions.
    1. Gets merge signature from preliminary shotdata and kinematic
    position arrays
    2. Checks if refinement is needed (via override or merge status)
    3. Merges shotdata with interpolated kinematic positions
    4. Writes refined shotdata to final TileDB array
    5. Records merge job in asset catalog

5. `process_svp`: Processes CTD and Seabird files to generate sound velocity profiles (SVP).
    
**Parameters:**
- `job='intermediate'`: Runs all intermediate processing stages sequentially. You can also specify individual stages like `run_pride`, `process_kinematic`, `process_dfop00`, `refines_shotdata`, `process_svp`
- `primary_config=global_config`: (optional) Uses the configuration settings defined above 

**Processing time:** This can take some time depending on campaign length. Progress will be displayed as each stage completes.

## Option 1: Run all in one command

In [ ]:
workflow.preprocess_run_pipeline_sv3(job="intermediate", 
                                     primary_config=global_config)

## Option 2: Run intermediate pipeline steps in order one by one

In [ ]:
workflow.preprocess_run_pipeline_sv3(job="run_pride",
                                     primary_config=global_config)

In [ ]:
workflow.preprocess_run_pipeline_sv3(job="process_kinematic",
                                     primary_config=global_config)

In [ ]:
workflow.preprocess_run_pipeline_sv3(job="process_dfop00",
                                     primary_config=global_config)

In [ ]:
workflow.preprocess_run_pipeline_sv3(job="refine_shotdata",
                                     primary_config=global_config)

In [ ]:
workflow.preprocess_run_pipeline_sv3(job="process_svp",
                                     primary_config=global_config)

### You are ready to move on to garpos processing! 